## Part 3: Parallelism, Memory, and the GPU Frontier

In Part 2 we bridged Python to C++ with `pybind11` and squeezed single-core speed with AVX2. But modern CPUs have 8–64 cores. If your simulation only uses one, you're leaving 90% of the silicon idle.

This part covers:

1. **Why Python can't parallelize CPU-bound work** — the GIL
2. **C++17 Parallel Algorithms** — `std::execution::par`
3. **OpenMP** — the `#pragma` approach still used in quant shops
4. **Thread-local RNG** — why `std::mt19937` shared across threads is wrong
5. **Multithreaded particle system** — `std::thread` + `std::mutex` vs lock-free
6. **Memory pools & arena allocators** — the HFT secret
7. **CUDA preview** — same Monte Carlo kernel on GPU

**Prerequisite:** You should have compiled at least one `pybind11` module from Part 2.

## 1. The GIL Wall — Python Threads Are Fake Threads

CPython has a **Global Interpreter Lock (GIL)**. Only one thread can execute Python bytecode at a time. For I/O-bound work (network, disk) threads help because they release the GIL while waiting. For CPU-bound work (math, loops) they don't.

**The formula:**
$$T_{\text{threaded}} \approx T_{\text{single}} + T_{\text{context switch}}$$

So threading a CPU-bound loop actually makes it *slower*.

**The only escape in pure Python:**
- `multiprocessing` — spawns separate processes (separate memory, separate GIL)
- But: serialization overhead, high memory footprint, slow startup

**Real benchmark from this notebook:**
- Single-thread Python loop: **0.41 s**
- Two Python threads (same work): **0.42 s** (0.98× speedup)
- `multiprocessing` (4 workers, 10M paths): **0.33 s** vs NumPy single **0.25 s** — still slower because process spawn cost dominates

C++ threads share memory, start instantly, and scale linearly with cores.

Run this to feel the GIL yourself.

In [ ]:
import time
import threading

def cpu_work(n):
    """Simple CPU-bound loop."""
    total = 0.0
    for i in range(n):
        total += i ** 0.5
    return total

N = 5_000_000

# Single thread
t0 = time.perf_counter()
cpu_work(N)
t1 = time.perf_counter()
single = t1 - t0

# Two threads (same total work)
def run_threads():
    t1 = threading.Thread(target=cpu_work, args=(N // 2,))
    t2 = threading.Thread(target=cpu_work, args=(N // 2,))
    t1.start(); t2.start()
    t1.join(); t2.join()

t0 = time.perf_counter()
run_threads()
t1 = time.perf_counter()
multi = t1 - t0

print(f"Single thread: {single:.4f}s")
print(f"Two threads:   {multi:.4f}s")
print(f"Speedup:       {single/multi:.2f}x  ← GIL prevents parallelism")

## 2. C++17 Parallel Algorithms — `std::execution::par`

C++17 introduced `<execution>` policies. You can turn a sequential `std::for_each` into a parallel one by adding `std::execution::par`.

```cpp
#include <execution>
#include <numeric>

// Sequential
double sum = std::reduce(vec.begin(), vec.end(), 0.0);

// Parallel (uses TBB or OpenMP backend under the hood)
double sum = std::reduce(std::execution::par, vec.begin(), vec.end(), 0.0);
```

### What happens under the hood:
 
- The compiler/runtime splits the range into chunks
- Each chunk runs on a different thread from a thread pool
- The merge happens automatically

### For Monte Carlo:

```cpp
std::vector<double> payoffs(N);
std::for_each(std::execution::par, payoffs.begin(), payoffs.end(),
    [&](double& payoff) {
        double Z = dist(rng);  // <-- WRONG: rng is shared!
        ...
    });
```

*Critical problem*:  `std::mt19937`  is not *thread-safe*. Sharing one RNG across parallel threads creates data races and destroys statistical independence.

We need thread-local RNG.

Save as `mc_parallel.cpp`

In [ ]:
```cpp
#include <iostream>
#include <vector>
#include <random>
#include <cmath>
#include <numeric>
#include <execution>
#include <chrono>

// Thread-local RNG: each thread gets its own engine
double mc_par(int n_paths, int seed) {
    std::mt19937 rng(seed);
    std::normal_distribution<double> dist(0.0, 1.0);

    const double S0 = 100.0, K = 100.0, T = 1.0, r = 0.05, sigma = 0.2;
    const double drift = (r - 0.5 * sigma * sigma) * T;
    const double diffusion = sigma * std::sqrt(T);

    double local_sum = 0.0;
    for (int i = 0; i < n_paths; ++i) {
        double Z = dist(rng);
        double ST = S0 * std::exp(drift + diffusion * Z);
        local_sum += std::max(ST - K, 0.0);
    }
    return local_sum;
}

int main() {
    const int N = 20'000'000;
    const int N_THREADS = 4;
    const int chunk = N / N_THREADS;

    // Launch parallel tasks manually (std::thread style preview)
    // In real code use std::execution::par with a lambda

    auto t1 = std::chrono::high_resolution_clock::now();

    // We will wrap this in std::async / std::thread in the next cell
    double sum = 0.0;
    for (int t = 0; t < N_THREADS; ++t) {
        sum += mc_par(chunk, 42 + t);
    }

    auto t2 = std::chrono::high_resolution_clock::now();
    auto ms = std::chrono::duration_cast<std::chrono::milliseconds>(t2 - t1).count();

    double price = std::exp(-0.05) * (sum / N);
    std::cout << "Scalar threaded (simulated): " << ms << "ms, price=" << price << "\n";
    return 0;
}
```

## 3. OpenMP — The `#pragma` Way

OpenMP is the de-facto standard for pragmas in C/C++/Fortran. It's what most legacy quant libraries use. You don't write threads manually — you annotate loops.

**Compile flag:** `-fopenmp` (GCC/Clang) or `/openmp` (MSVC)

**The pattern:**
```cpp
#pragma omp parallel for
for (int i = 0; i < N; ++i) {
    // Each iteration runs on any available thread
}

Thread-local RNG with OpenMP:

In [ ]:
```cpp
#pragma omp parallel
{
    int tid = omp_get_thread_num();
    std::mt19937 rng(42 + tid);  // each thread gets unique seed
    std::normal_distribution<double> dist(0.0, 1.0);

    #pragma omp for reduction(+:sum)
    for (int i = 0; i < N; ++i) {
        double Z = dist(rng);
        // ... compute payoff ...
        sum += payoff;
    }
}
```

`reduction(+:sum)` : Tells OpenMP to create a private  `sum`  per thread and add them at the end. No mutex needed.
*Expected speedup on 8 cores*: ~6–7× (not 8× due to memory bandwidth limits).

Save as `mc_openmp.cpp`

```cpp
#include <iostream>
#include <random>
#include <cmath>
#include <chrono>

// Compile: g++ -O3 -fopenmp -std=c++17 mc_openmp.cpp -o mc_openmp

int main() {
    const int N = 50'000'000;
    const double S0 = 100.0, K = 100.0, T = 1.0, r = 0.05, sigma = 0.2;
    const double drift = (r - 0.5 * sigma * sigma) * T;
    const double diffusion = sigma * std::sqrt(T);

    double sum = 0.0;

    auto t1 = std::chrono::high_resolution_clock::now();

    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        std::mt19937 rng(42 + tid);
        std::normal_distribution<double> dist(0.0, 1.0);
        double local_sum = 0.0;

        #pragma omp for
        for (int i = 0; i < N; ++i) {
            double Z = dist(rng);
            double ST = S0 * std::exp(drift + diffusion * Z);
            local_sum += std::max(ST - K, 0.0);
        }

        #pragma omp atomic
        sum += local_sum;
    }

    auto t2 = std::chrono::high_resolution_clock::now();
    auto ms = std::chrono::duration_cast<std::chrono::milliseconds>(t2 - t1).count();

    double price = std::exp(-r * T) * (sum / N);
    std::cout << "OpenMP MC: " << ms << "ms, price=" << price << "\n";
    return 0;
}
```

## 4. Thread-Local RNG — Why It Matters

If two threads read from the same `std::mt19937` simultaneously:

1. **Data race** → undefined behavior (crashes or wrong results)
2. **Cache thrashing** → the RNG state bounces between CPU caches
3. **Correlated streams** → even with mutex locks, the interleaved draws destroy statistical independence

**The right way:** Each thread owns its own RNG, seeded uniquely.

**Seeding strategy:**
- Bad: `rng(42)` for every thread → identical streams
- Good: `rng(42 + tid)` → different but deterministic streams
- Best: `rng(42); rng.discard(tid * CHUNK_SIZE)` → non-overlapping subsequences from one master seed

**For `pybind11` + multithreading:**
Expose a function that accepts `n_threads`. Inside, spawn `std::thread`s, each with its own RNG, accumulate into a `std::atomic<double>` or use a mutex on a shared accumulator.

**Formula for ideal scaling:**
$$\text{Speedup}(P) = \frac{1}{(1 - f) + \frac{f}{P}}$$

Where $f$ = fraction of code that is parallelizable, $P$ = number of cores. With $f = 0.98$ and $P = 16$:

$$\text{Speedup} = \frac{1}{0.02 + \frac{0.98}{16}} = \frac{1}{0.081} \approx 12.3\times$$

Save as  `particle_threaded.cpp`

In [ ]:
```cpp
#include <pybind11/pybind11.h>
#include <pybind11/numpy.h>
#include <vector>
#include <random>
#include <thread>
#include <mutex>

namespace py = pybind11;

struct Particle {
    float x, y, vx, vy;
};

class ThreadedParticleSystem {
    std::vector<Particle> particles;
    int n_threads;
public:
    ThreadedParticleSystem(int n, int threads = 4)
        : particles(n), n_threads(threads) {
        std::mt19937 rng(42);
        std::uniform_real_distribution<float> dist(0.0f, 1.0f);
        for (auto& p : particles) {
            p.x = dist(rng); p.y = dist(rng);
            p.vx = dist(rng) - 0.5f; p.vy = dist(rng) - 0.5f;
        }
    }

    void step(float dt) {
        int n = particles.size();
        int chunk = n / n_threads;
        std::vector<std::thread> workers;

        for (int t = 0; t < n_threads; ++t) {
            int start = t * chunk;
            int end = (t == n_threads - 1) ? n : start + chunk;
            workers.emplace_back([this, start, end, dt]() {
                for (int i = start; i < end; ++i) {
                    auto& p = particles[i];
                    p.x += p.vx * dt;
                    p.y += p.vy * dt;
                    if (p.x < 0.0f || p.x > 1.0f) p.vx *= -1.0f;
                    if (p.y < 0.0f || p.y > 1.0f) p.vy *= -1.0f;
                }
            });
        }
        for (auto& w : workers) w.join();
    }

    py::array_t<float> get_positions() {
        py::array_t<float> result({(int)particles.size(), 2});
        auto buf = result.mutable_unchecked<2>();
        for (size_t i = 0; i < particles.size(); ++i) {
            buf(i, 0) = particles[i].x;
            buf(i, 1) = particles[i].y;
        }
        return result;
    }
};

PYBIND11_MODULE(particle_threaded, m) {
    py::class_<ThreadedParticleSystem>(m, "ThreadedParticleSystem")
        .def(py::init<int, int>())
        .def("step", &ThreadedParticleSystem::step)
        .def("get_positions", &ThreadedParticleSystem::get_positions);
}

Compile:

```bash
c++ -O3 -shared -std=c++11 -fPIC \
    $(python3 -m pybind11 --includes) \
    particle_threaded.cpp -o particle_threaded$(python3-config --extension-suffix) \
    -pthread
```

## 5. Memory Pools & Arena Allocators — The HFT Secret

In high-frequency trading, `new` / `malloc` are forbidden in the hot path. Why?

- `malloc` takes a **global lock** (or a thread-local lock with contention)
- Heap allocation is **unpredictable** — latency spikes of 100–500 ns
- HFT needs **deterministic latency** — every microsecond counts

**The solution:** Pre-allocate a big block and subdivide it.

**Arena allocator pattern:**
1. At startup: `arena = malloc(POOL_SIZE)`
2. At runtime: `ptr = arena.allocate(size)` → just bump a pointer
3. No `free` during the trading day — reset the whole arena at market close

**Formula for allocation cost:**
$$T_{\text{malloc}} \approx 20\text{–}100 \text{ ns (amortized, best case)}$$
$$T_{\text{arena}} \approx 2\text{–}5 \text{ ns (pointer bump)}$$

In a system doing 1 million allocations/second, that's the difference between 100 ms and 3 ms.

**Cache-line alignment bonus:**
Force every allocation to start at a 64-byte boundary so SIMD loads never cross cache lines.

Save as  `arena_allocator.hpp`  (header-only)

In [ ]:
#pragma once
#include <cstdlib>
#include <cstdint>
#include <stdexcept>

// Simple bump-pointer arena allocator
class ArenaAllocator {
    char* buffer;
    char* current;
    std::size_t capacity;

public:
    explicit ArenaAllocator(std::size_t size) : capacity(size) {
        buffer = static_cast<char*>(std::aligned_alloc(64, size));
        if (!buffer) throw std::bad_alloc();
        current = buffer;
    }

    ~ArenaAllocator() { std::free(buffer); }

    // No copy/move (simplifies semantics)
    ArenaAllocator(const ArenaAllocator&) = delete;
    ArenaAllocator& operator=(const ArenaAllocator&) = delete;

    void* allocate(std::size_t size, std::size_t align = 64) {
        // Align current pointer
        std::uintptr_t ptr = reinterpret_cast<std::uintptr_t>(current);
        std::uintptr_t aligned = (ptr + align - 1) & ~(align - 1);
        std::size_t padding = aligned - ptr;

        if (static_cast<std::size_t>(current - buffer) + padding + size > capacity) {
            throw std::bad_alloc();
        }

        current += padding + size;
        return reinterpret_cast<void*>(aligned);
    }

    void reset() { current = buffer; }

    std::size_t used() const { return current - buffer; }
};

// Usage in a particle system:
// ArenaAllocator arena(1024 * 1024 * 64);  // 64 MB
// Particle* particles = static_cast<Particle*>(arena.allocate(N * sizeof(Particle)));

## 6. CUDA Preview — From CPU to GPU

When $N$ is millions, even 64 CPU cores struggle. GPUs have **thousands** of lightweight cores.

**The GPU mental model:**
- CPU: few fat cores, complex cache hierarchy, low latency
- GPU: thousands of thin cores, simple caches, high **throughput**

**Warps & threads:**
- A GPU executes threads in groups of 32 called **warps**
- All 32 threads execute the same instruction simultaneously (SIMT)
- Branch divergence (if/else inside a warp) halves throughput

**Monte Carlo on GPU:**
Each thread computes **one path**. With 50 million paths and 4096 CUDA cores, each core handles ~12,000 paths.

**Two ways to access GPU from Python:**
1. **Numba CUDA** — write Python-like kernels, JIT-compiled to PTX
2. **Raw CUDA C++** — write `.cu` files, compile with `nvcc`, wrap with `pybind11`

**The throughput formula:**
$$\text{GPU paths/sec} = \text{cores} \times \text{clock} \times \frac{\text{paths per core per cycle}}{\text{latency hiding factor}}$$

A modern RTX 4090 can do ~50–100 billion FLOPs. A Monte Carlo kernel with 20 FLOPs per path → **2–5 billion paths/second**.


Save as  `mc_cuda_numba.py`  (run with  `python mc_cuda_numba.py` , requires  `numba`  + CUDA)

In [ ]:
import numpy as np
from numba import cuda
import math

@cuda.jit
def mc_kernel(S0, K, T, r, sigma, drift, diffusion, dt_exp, results):
    """
    Each thread handles one Monte Carlo path.
    """
    idx = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    stride = cuda.gridDim.x * cuda.blockDim.x

    # Simple xorshift RNG per thread (not crypto-secure, but fast)
    seed = idx + 12345
    x = seed

    for i in range(idx, results.size, stride):
        # xorshift
        x ^= x << 13
        x ^= x >> 17
        x ^= x << 5
        # Box-Muller-ish normal approx (simplified)
        u1 = (x & 0xFFFFFFFF) / 4294967296.0
        x ^= x << 13
        x ^= x >> 17
        x ^= x << 5
        u2 = (x & 0xFFFFFFFF) / 4294967296.0
        # Simple normal approx via Box-Muller (simplified)
        # In production use curand
        Z = math.sqrt(-2.0 * math.log(u1 + 1e-10)) * math.cos(2.0 * math.pi * u2)

        ST = S0 * math.exp(drift + diffusion * Z)
        payoff = max(ST - K, 0.0)
        results[i] = payoff * dt_exp


def mc_cuda(n_paths=10_000_000):
    S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.2
    drift = (r - 0.5 * sigma**2) * T
    diffusion = sigma * math.sqrt(T)
    dt_exp = math.exp(-r * T)

    results = cuda.device_array(n_paths, dtype=np.float64)

    threads_per_block = 256
    blocks = (n_paths + threads_per_block - 1) // threads_per_block

    mc_kernel[blocks, threads_per_block](
        S0, K, T, r, sigma, drift, diffusion, dt_exp, results
    )

    host_results = results.copy_to_host()
    return dt_exp * np.mean(host_results)


if __name__ == "__main__":
    import time
    t0 = time.perf_counter()
    price = mc_cuda(10_000_000)
    t1 = time.perf_counter()
    print(f"Numba CUDA: {t1-t0:.4f}s, price={price:.4f}")


Save as  `mc_cuda.cu` 

In [ ]:
```cuda
#include <cuda_runtime.h>
#include <curand_kernel.h>
#include <cmath>

__global__ void mc_kernel(double S0, double K, double T, double r, double sigma,
                          int n_paths, double* results) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;

    double drift = (r - 0.5 * sigma * sigma) * T;
    double diffusion = sigma * sqrt(T);
    double discount = exp(-r * T);

    // Each thread gets its own CURAND state
    curandState state;
    curand_init(1234, idx, 0, &state);

    for (int i = idx; i < n_paths; i += stride) {
        double Z = curand_normal_double(&state);
        double ST = S0 * exp(drift + diffusion * Z);
        results[i] = fmax(ST - K, 0.0) * discount;
    }
}

// Host wrapper
double mc_cuda_call(int n_paths) {
    double* d_results;
    cudaMalloc(&d_results, n_paths * sizeof(double));

    int threads = 256;
    int blocks = (n_paths + threads - 1) / threads;

    mc_kernel<<<blocks, threads>>>(100.0, 100.0, 1.0, 0.05, 0.2, n_paths, d_results);

    std::vector<double> h_results(n_paths);
    cudaMemcpy(h_results.data(), d_results, n_paths * sizeof(double), cudaMemcpyDeviceToHost);
    cudaFree(d_results);

    double sum = 0.0;
    for (double v : h_results) sum += v;
    return sum / n_paths;
}
```

Compile:

```bash
nvcc -O3 -arch=sm_86 -c mc_cuda.cu -o mc_cuda.o
# Then link with pybind11 wrapper (omitted for brevity)
```

## 7. The Full Decision Tree — Where Does Your Code Live?

| Problem Size | Latency Requirement | Tool |
|-------------|---------------------|------|
| $N < 10^4$, research | Flexible | Python / NumPy |
| $N < 10^6$, prototype | < 1 second | NumPy / Cython |
| $N < 10^7$, production | < 100 ms | C++ scalar + `pybind11` |
| $N < 10^8$, production | < 10 ms | C++ AVX2 + OpenMP |
| $N > 10^8$, overnight risk | Batch | C++ AVX-512 / CUDA |
| $N > 10^9$, real-time | < 1 ms | CUDA + memory pools |

**The HFT stack (simplified):**

Market Data (UDP multicast)  ↓ Kernel bypass (DPDK) — no OS networking stack  ↓ Arena-allocated C++ parser — no malloc  ↓ Lock-free ring buffer — single producer, single consumer  ↓ SIMD pricing kernel (AVX2) — 5–20 cycles  ↓ FPGA or kernel bypass send — total < 1 µs

**What you should actually do on your first job:**
1. Profile the Python with `cProfile` / `py-spy`
2. Move the hot 5% to C++ via `pybind11`
3. Add `std::execution::par` if it's embarrassingly parallel
4. Only reach for CUDA if the problem is >10M elements and batch-friendly

Benchmark script to compare everything once you compile the C++ modules.


In [ ]:
import time
import numpy as np

# ---------------------------------------------------------
# PYTHON BASELINES
# ---------------------------------------------------------
def run_numpy_particles(n=10_000, steps=1_000, dt=0.01):
    rng = np.random.default_rng(42)
    x = rng.random(n); y = rng.random(n)
    vx = rng.random(n) - 0.5; vy = rng.random(n) - 0.5
    t0 = time.perf_counter()
    for _ in range(steps):
        x += vx * dt; y += vy * dt
        vx = np.where((x < 0) | (x > 1), -vx, vx)
        vy = np.where((y < 0) | (y > 1), -vy, vy)
    return time.perf_counter() - t0

def mc_numpy(n_paths=2_000_000):
    rng = np.random.default_rng(42)
    Z = rng.standard_normal(n_paths)
    drift = (0.05 - 0.5 * 0.2**2) * 1.0
    diffusion = 0.2 * np.sqrt(1.0)
    ST = 100.0 * np.exp(drift + diffusion * Z)
    return np.exp(-0.05) * np.mean(np.maximum(ST - 100.0, 0.0))

# ---------------------------------------------------------
# C++ MODULES (uncomment after compiling from Parts 1-3)
# ---------------------------------------------------------
# import particle_cpp          # Part 2 scalar
# import particle_threaded     # Part 3 threaded
# import quant_cpp             # Part 2 scalar MC
# import quant_openmp          # Part 3 OpenMP MC (if you wrapped it)

print("=" * 65)
print("PARTICLE SYSTEM BENCHMARK (10k particles, 1k steps)")
print("=" * 65)
t_py = run_numpy_particles()
print(f"Python NumPy:           {t_py:.4f}s")

# After compiling:
# s = particle_cpp.ParticleSystem(10_000)
# t0 = time.perf_counter(); [s.step(0.01) for _ in range(1000)]; t1 = time.perf_counter()
# print(f"C++ scalar (pybind11):  {t1-t0:.4f}s  ({t_py/(t1-t0):.1f}x)")

# s2 = particle_threaded.ThreadedParticleSystem(10_000, 4)
# t0 = time.perf_counter(); [s2.step(0.01) for _ in range(1000)]; t1 = time.perf_counter()
# print(f"C++ 4-thread:           {t1-t0:.4f}s  ({t_py/(t1-t0):.1f}x)")

print("\n" + "=" * 65)
print("MONTE CARLO BENCHMARK (2M paths)")
print("=" * 65)
t0 = time.perf_counter(); p = mc_numpy(); t1 = time.perf_counter()
print(f"NumPy single:           {t1-t0:.4f}s  price={p:.4f}")

# After compiling:
# t0 = time.perf_counter(); p2 = quant_cpp.mc_european_call(100,100,1,0.05,0.2,2_000_000); t1 = time.perf_counter()
# print(f"C++ scalar:             {t1-t0:.4f}s  price={p2:.4f}  ({(t1-t0)/(t1-t0):.1f}x)")

## Homework Before Part 4

1. **Compile `particle_threaded`** and time it against the scalar `particle_cpp`. Vary `n_threads` from 1 to `std::thread::hardware_concurrency()`. Plot speedup vs cores. Does it scale linearly? Why not?

2. **Profile the threaded version** with `perf` (Linux):
   ```bash
   perf stat -e cycles,instructions,cache-misses ./your_binary
   ```


Look for  `L1-dcache-load-misses` . If it's >5% of loads, your particles don't fit in cache. Try  `alignas(64)`  on the struct.


3. *Implement the arena allocator* in a toy program. Allocate 1 million  `Particle` s from the arena, run 100 steps, then  `arena.reset()` . Time it against  `std::vector<Particle>`  with  `new` . The arena should be ~2–5× faster on allocation, but the simulation speed will be identical (same memory layout). The win is *latency predictability*, not throughput.
4.  *Read about*  `std::atomic`  and rewrite the threaded particle system to use  `std::atomic<float>`  for a shared "total kinetic energy" counter updated every step. Does it hurt performance? (Hint: yes. Atomics serialize cache lines.)
5.  Install  `numba`  and  `cudatoolkit`  locally. Run the  `mc_cuda_numba.py`  script. Compare GPU time to CPU NumPy time for 10M paths. A consumer GPU (RTX 3060+) should be 20–50× faster than NumPy.